# BIST 100 Exploratory Data Analysis

This notebook examines the validated daily OHLCV history for the BIST 100 index (`XU100.IS`) before forecasting.

The goals are to:

- inspect the available date range and data quality;
- summarize closing values and daily returns;
- measure volatility and historical drawdown;
- visualize the index level and daily percentage changes.

Visible historical patterns are not treated as evidence of future predictability.

In [ ]:
from pathlib import Path

import pandas as pd

from bist100_forecasting.data import DEFAULT_DATA_PATH, load_history
from bist100_forecasting.eda import (
    build_eda_figure,
    calculate_daily_returns,
    summarize_history,
)

pd.options.display.float_format = "{:,.4f}".format
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / DEFAULT_DATA_PATH
DATA_PATH

In [ ]:
history = load_history(DATA_PATH)
print(f"Loaded {len(history):,} observations from {DATA_PATH}")
history.tail()

## Data quality

The loader has already applied the project validation rules. The following table makes the most important checks visible in the notebook.

In [ ]:
quality_checks = pd.Series(
    {
        "Rows": len(history),
        "Columns": len(history.columns),
        "Missing values": int(history.isna().sum().sum()),
        "Duplicate dates": int(history.index.duplicated().sum()),
        "Dates sorted": history.index.is_monotonic_increasing,
        "First date": history.index.min().date(),
        "Last date": history.index.max().date(),
    },
    name="Value",
)
quality_checks.to_frame()

## Market history summary

Returns are close-to-close percentage changes. Maximum drawdown is the largest historical decline from a previous closing-value peak.

In [ ]:
summary = summarize_history(history)
summary_table = pd.Series(
    {
        "Observations": summary.observations,
        "First close": summary.first_close,
        "Latest close": summary.latest_close,
        "Minimum close": summary.minimum_close,
        "Maximum close": summary.maximum_close,
        "Total return (%)": summary.total_return_pct,
        "Mean daily return (%)": summary.mean_daily_return_pct,
        "Daily volatility (%)": summary.daily_volatility_pct,
        "Maximum drawdown (%)": summary.maximum_drawdown_pct,
    },
    name="Value",
)
summary_table.to_frame()

In [ ]:
history.describe().T

In [ ]:
daily_returns = calculate_daily_returns(history)
daily_returns.mul(100).describe().to_frame("Daily return (%)")

## Closing value and daily return

The first panel shows the index level. The second panel shows daily percentage changes around zero.

In [ ]:
figure = build_eda_figure(history)
figure

## Next step

The modeling pipeline will preserve chronological order when splitting the data. Scaling parameters will be learned from the training period only. Validation and test targets will remain inside their own periods while their input windows use only observations available before each target date.